In [1]:
import pandas as pd
import numpy as np
from scipy import linalg

In [26]:
# get amount values to array
df = pd.read_csv("transactions.csv")
amounts = df["tsc_amt"].values

In [58]:
df["tsc_dt"] = pd.to_datetime(df["tsc_dt"])
df["week"] = df["tsc_dt"].dt.isocalendar().week
# Remove rows where tsc_type is "CDT" or tsc_descrp is "income"
df_clean = df[
    (df["tsc_type"].fillna("").str.upper() != "CDT") &
    (df["tsc_descrp"].fillna("").str.lower() != "income")
]

In [60]:

matrix = df_clean.pivot_table(
    index="week",
    columns="tsc_cat",
    values="tsc_amt",
    aggfunc="sum",
    fill_value=0
)
week_map = {old: new for new, old in enumerate(matrix.index, start=1)}
matrix.index = matrix.index.map(week_map)
X = matrix.to_numpy()
matrix

tsc_cat,Clothing,Food,Groceries,Loan,Miscellaneous,Transportation
week,,,,,,
1,102.2,80.3,1.3,0.0,0.0,110.00
2,0.0,8.8,0.0,0.0,0.0,110.00
3,0.0,31.2,1.2,0.0,0.0,179.99
4,0.0,87.2,105.3,0.0,0.0,89.68
5,0.0,14.0,9.4,400.0,30.5,299.53


In [ ]:
weekly_spending = X.sum(axis=1)
category_spending = X.sum(axis=0)

Weekly Spending: [293.8  118.8  212.39 282.18 753.43]


In [57]:
result_cat = pd.DataFrame({
    "Category": matrix.columns,
    "Category_Spending": category_spending,
})
result_cat.sort_values("Category_Spending", ascending=False)

,Category,Category_Spending
5,Transportation,789.2
3,Loan,400.0
1,Food,221.5
2,Groceries,117.2
0,Clothing,102.2
4,Miscellaneous,30.5


In [ ]:
result_df = pd.DataFrame({
    "Week": matrix.index,
    "Weekly_spending": weekly_spending,
})
result_df

,Week,Weekly_spending
0,1,293.80
1,2,118.80
2,3,212.39
3,4,282.18
4,5,753.43
